# Video2World · Quick Eval

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AetherLabsAI/Video2World/blob/main/notebooks/v2w_colab.ipynb)

Choose a mode in the first cell and press **Runtime → Run all**. One instance is evaluated, so a run stays short.

| Mode | What happens | You need |
|---|---|---|
| **try it** | Scores the human-assisted reference (~1 min) to show what a result looks like. | nothing (CPU runtime is fine) |
| **my package** | Evaluates a protocol package you made: upload a `.zip` with `<family>/<sample>/protocol.json`. | GPU runtime |
| **run an agent** | A coding agent builds the world from the video, then it is evaluated and scored. Up to 3 h. | GPU runtime + an API key in 🔑 *Secrets* |

Colab covers the ManiSkill and MuJoCo tracks (160 of 222 instances). RoboDojo and in-house instances need Isaac Sim on your own GPU machine, as does the full V2WScore: see the README. Your own agent command or a self-hosted endpoint: `docs/custom_agent.md`.

In [ ]:
#@title 1 · Choose
MODE = "try it"  #@param ["try it", "my package", "run an agent"]
TASK = "push-t"  #@param {type:"string"}
#@markdown A task family (`push-t`, `drawer-insert`, ...) or `family/sample`. Run `v2w list` in a cell to see all. Ignored by **try it**; for **my package**, empty uses the first package in your zip.
MODEL = "claude-opus-5-5"  #@param {type:"string"}
#@markdown Only for **run an agent**. `claude-*` runs Claude Code (key `ANTHROPIC_API_KEY`), `gpt-*` runs Codex (`OPENAI_API_KEY`), anything like `moonshotai/kimi-k3` goes through OpenRouter (`OPENROUTER_API_KEY`). Add the key under 🔑 Secrets and enable notebook access.

In [ ]:
#@title 2 · Set up (automatic)
import os, shutil, shlex, json, zipfile
from pathlib import Path
RUN_NAME = "colab"
REPO = "AetherLabs-AI/Video2World"
def sh(cmd): return get_ipython().system(cmd)

if not os.path.isdir("/content/Video2World"):
    sh("git clone -q --depth 1 https://github.com/AetherLabsAI/Video2World.git /content/Video2World")
os.chdir("/content/Video2World")
sh("pip install -q -e . huggingface_hub")
from huggingface_hub import snapshot_download
def fetch(patterns): snapshot_download(REPO, repo_type="dataset", local_dir="data", allow_patterns=patterns)

if MODE == "try it":
    REPORT = "runs/har_report"
    fetch(["manifest.json", "paths.json", "har/**", "LICENSE"])
else:
    import torch
    if not torch.cuda.is_available():
        raise SystemExit("This mode needs a GPU: Runtime → Change runtime type → T4 GPU, then Run all again.")
    sh("apt-get -qq install -y libvulkan1 mesa-vulkan-drivers libgl1 >/dev/null")
    sh("pip install -q -e .[sim] && python -m mani_skill.utils.download_asset xarm6 -y")
    sh("[ -d .venv-twin ] || (python -m venv .venv-twin && .venv-twin/bin/pip install -q -e .[twin]); v2w setup --twin-python .venv-twin/bin/python")
    fetch(["manifest.json", "paths.json", "har/**", "assets/**", "sources/**", "LICENSE"])

    from v2w import benchmark
    runnable = [c for c in benchmark.configurations() if c.track in {"furniture", "hand", "twins", "cloth"}]
    if MODE == "my package":
        from google.colab import files
        name = next(iter(files.upload()))
        PKG = "/content/packages"; shutil.rmtree(PKG, ignore_errors=True); zipfile.ZipFile(name).extractall(PKG)
        have = {(p.parent.parent.name, p.parent.name) for p in Path(PKG).glob("*/*/protocol.json")}
        runnable = [c for c in runnable if (c.family, c.sample) in have]
    fam, _, smp = TASK.strip().partition("/")
    chosen = [c for c in runnable if (not fam or c.family == fam) and (not smp or c.sample == smp)]
    if not chosen:
        raise SystemExit(f"No runnable instance for TASK={TASK!r}. Colab runs ManiSkill/MuJoCo families only (not robodojo-* or inhouse-*).")
    ONE = chosen[0]
    fetch([f"samples/{ONE.sample}/**"])
    REPORT = f"runs/{RUN_NAME}/report"
    print("Evaluating:", ONE.key)

In [ ]:
#@title 3 · Run
if MODE == "try it":
    sh("v2w har --out runs/har_report")
else:
    one = f"--name {RUN_NAME} --video --family {shlex.quote(ONE.family)} --sample {shlex.quote(ONE.sample)}"
    if MODE == "my package":
        sh(f"v2w eval {one} --packages /content/packages")
    else:
        if MODEL.startswith("claude"): agent, secret, extra = "claude", "ANTHROPIC_API_KEY", ""
        elif MODEL.startswith(("gpt", "o1", "o3", "o4")): agent, secret, extra = "codex", "OPENAI_API_KEY", ""
        else: agent, secret, extra = "opencode", "OPENROUTER_API_KEY", " --base-url https://openrouter.ai/api/v1 --api-key-env OPENROUTER_API_KEY"
        try:
            from google.colab import userdata; os.environ[secret] = userdata.get(secret)
        except Exception:
            from getpass import getpass; os.environ[secret] = getpass(f"{secret}: ")
        sh("npm install -g --silent @anthropic-ai/claude-code @openai/codex opencode-ai")
        sh(f"v2w run {one} --model {shlex.quote(MODEL)}" + (f" --agent {agent}" if agent != "opencode" else "") + extra)

In [ ]:
#@title 4 · Results
import base64, glob, html
import matplotlib.pyplot as plt
from IPython.display import HTML, display

INK, MUTED = "#2b2118", "#8a7d70"
DIM = {"F": ("#4f6f7d", "Function"), "G": ("#7c9a6d", "Geometry"), "D": ("#b0794f", "Dynamics")}
plt.rcParams.update({"figure.facecolor": "white", "axes.facecolor": "white", "axes.edgecolor": MUTED, "text.color": INK,
                     "axes.labelcolor": INK, "xtick.color": MUTED, "ytick.color": INK, "font.size": 10})

summary = json.load(open(f"{REPORT}/summary.json"))
rows = json.load(open(f"{REPORT}/samples.json"))

def card(title, subtitle, score, parts):
    chips = "".join(f'<div style="flex:1"><div style="font:12px sans-serif;color:{MUTED}">{DIM[k][1]}</div>'
                    f'<div style="font:600 24px serif;color:{DIM[k][0]}">{v:.2f}</div></div>' for k, v in parts.items())
    display(HTML(f'<div style="border:1px solid #d9cfc3;border-radius:12px;padding:18px 22px;max-width:640px;background:#faf7f2;color:{INK}">'
                 f'<div style="font:12px sans-serif;color:{MUTED};letter-spacing:.08em;text-transform:uppercase">{html.escape(subtitle)}</div>'
                 f'<div style="font:600 18px sans-serif;margin:2px 0 10px">{html.escape(title)}</div>'
                 f'<div style="display:flex;gap:24px;align-items:flex-end"><div><div style="font:12px sans-serif;color:{MUTED}">Score</div>'
                 f'<div style="font:600 44px serif;line-height:1">{score:.1f}</div></div>{chips}</div></div>'))

def video_html(path, caption):
    if not path or not os.path.exists(path) or os.path.getsize(path) > 25e6:
        return ""
    b64 = base64.b64encode(open(path, "rb").read()).decode()
    return (f'<figure style="margin:0"><video src="data:video/mp4;base64,{b64}" width="360" controls loop muted></video>'
            f'<figcaption style="font:12px sans-serif;color:{MUTED}">{caption}</figcaption></figure>')

if MODE == "try it":
    card("Human-assisted reference", f"{summary['configurations']} instances · {summary['families']} families",
         summary["v2w_score"], {k: summary["dimensions"][k] for k in "FGD"} if all(k in summary["dimensions"] for k in "FGD") else {})
    fam = sorted(summary["by_family"].items(), key=lambda kv: kv[1]["score"])
    names = [k for k, _ in fam]; vals = [v["score"] for _, v in fam]
    fig, ax = plt.subplots(figsize=(7, 0.22 * len(names) + 1))
    ax.barh(names, vals, color="#5b4636", height=0.65); ax.set_xlim(0, 100); ax.set_xlabel("V2WScore")
    for sp in ("top", "right"): ax.spines[sp].set_visible(False)
    ax.set_title("Reference score per task family", loc="left", fontsize=11); plt.tight_layout(); plt.show()
else:
    r = next(x for x in rows if x["family"] == ONE.family and x["sample"] == ONE.sample)
    card(ONE.key, "one instance · try-out, not a leaderboard score", 100 * (r.get("S") or 0),
         {k: (r.get(k) or 0) for k in "FGD"})
    if not r.get("build"):
        print("The package did not build (build = false), so the score is 0. See the evaluator log under", f"runs/{RUN_NAME}/{ONE.family}/{ONE.sample}/")
    else:
        m = [(g, n, d) for g, grp in (("G", r.get("geometry") or {}), ("D", r.get("dynamics") or {})) for n, d in grp.items()]
        fig, ax = plt.subplots(figsize=(7, 0.45 * len(m) + 1.2))
        for i, (g, n, d) in enumerate(m):
            ax.barh(i, 1, color="#efe9e0", height=0.6)
            ax.barh(i, d["quality"], color=DIM[g][0], height=0.6)
            ax.text(1.02, i, f"{d['error']:.2f} / τ {d['tau']:g}", va="center", fontsize=9, color=MUTED)
        ax.set_yticks(range(len(m))); ax.set_yticklabels([n.replace("own_trajectory.", "") for _, n, _ in m]); ax.invert_yaxis()
        ax.set_xlim(0, 1); ax.set_xlabel("quality = max(0, 1 − error / τ)   (1 is perfect)")
        for sp in ("top", "right"): ax.spines[sp].set_visible(False)
        ax.set_title("Metric quality (green: geometry, orange: dynamics)", loc="left", fontsize=11); plt.tight_layout(); plt.show()
    src_video = str(Path("data/samples") / ONE.sample / "video.mp4")
    sim_video = next(iter(sorted(glob.glob(f"runs/{RUN_NAME}/{ONE.family}/{ONE.sample}/eval/video/*.mp4"))), None)
    display(HTML('<div style="display:flex;gap:16px;flex-wrap:wrap">' + video_html(src_video, "source video (the agent's only input)")
                 + video_html(sim_video, "your world, executed by the evaluator") + "</div>"))

shutil.make_archive("/content/v2w_report", "zip", REPORT)
try:
    from google.colab import files; files.download("/content/v2w_report.zip")
except Exception: pass